In [4]:
# ============================================================
# ICPR-TVRID 2026 — FIXED EFFICIENTNET-V2-L DUAL-STREAM
# ============================================================
!pip install timm pytorch-metric-learning -q

import os, random, math, gc
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from torch.amp import autocast, GradScaler

# ============================================================
# CONFIGURATION
# ============================================================
class CFG:
    root          = "/kaggle/input/extracted-dataset"
    train_base    = os.path.join(root, "train")
    train_csv     = os.path.join(root, "train_labels.csv") 

    backbone_name = "tf_efficientnetv2_l" 
    img_size      = 224
    embed_dim     = 512
    
    epochs         = 50
    batch_size     = 4     #
    accumulation_step = 2  # Effective batch size = 8
    lr             = 5e-5  
    val_ratio      = 0.20
    device         = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# UTILS & DATA (Same as before)
# ============================================================
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)

def normalize_path(p): return str(p).replace("\\", "/")

def build_train_index(base_path):
    index = {}
    base = Path(base_path)
    for passage_dir in base.glob("*/*/*"):
        if not passage_dir.is_dir(): continue
        rgb_files = sorted(passage_dir.glob("*RGB*.png"))
        depth_map = {d.name.replace("_depth.png", ""): d for d in passage_dir.glob("*depth*.png")}
        pairs = [(str(r), str(depth_map[r.name.replace("_RGB.png", "")])) 
                 for r in rgb_files if r.name.replace("_RGB.png", "") in depth_map]
        if pairs:
            parts = passage_dir.parts
            index[(parts[-3], parts[-2], parts[-1])] = pairs
    return index

# ============================================================
# MODEL: EFFICIENTNET V2-L (FIXED DIMENSIONS)
# ============================================================
class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super(GeM, self).__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), (x.size(-2), x.size(-1))).pow(1./self.p)

class TVRIDModelEffNetV2L(nn.Module):
    def __init__(self, num_ids):
        super().__init__()
        # Use features_only=True to get raw maps
        self.rgb_enc = timm.create_model(CFG.backbone_name, pretrained=True, features_only=True)
        self.dep_enc = timm.create_model(CFG.backbone_name, pretrained=True, features_only=True)
        
        self.gem = GeM()
        
        # ── FIXED DIMENSION: 640 ────────────────────────────
        # EfficientNetV2-L features_only mode outputs 640 at the final stage
        feat_dim = 640 
        
        self.dep_proj = nn.Sequential(nn.Linear(feat_dim, feat_dim), nn.LayerNorm(feat_dim))
        self.cross_attn = nn.MultiheadAttention(feat_dim, num_heads=8, batch_first=True)
        
        self.fusion = nn.Sequential(
            nn.Linear(feat_dim * 2, CFG.embed_dim),
            nn.BatchNorm1d(CFG.embed_dim),
            nn.GELU()
        )
        self.bnneck = nn.BatchNorm1d(CFG.embed_dim)
        self.classifier = nn.Linear(CFG.embed_dim, num_ids, bias=False)

    def encode(self, rgb, dep):
        # Extract last map [-1]
        f_rgb = self.gem(self.rgb_enc(rgb)[-1]).flatten(1)
        f_dep = self.gem(self.dep_enc(dep)[-1]).flatten(1)
        
        f_dep_p = self.dep_proj(f_dep)
        f_cross, _ = self.cross_attn(f_rgb.unsqueeze(1), f_dep_p.unsqueeze(1), f_dep_p.unsqueeze(1))
        
        feat = self.fusion(torch.cat([f_rgb, f_cross.squeeze(1)], dim=1))
        return self.bnneck(feat)

    def forward(self, rgb, dep):
        feat = self.encode(rgb, dep)
        return feat, self.classifier(feat)

# ============================================================
# PREPARE DATA & START TRAINING
# ============================================================
train_index = build_train_index(CFG.train_base)
train_df = pd.read_csv(CFG.train_csv)
samples = []
for _, row in train_df.iterrows():
    clean_path = normalize_path(row["path"])
    parts = clean_path.strip("/").split("/")
    if len(parts) >= 3:
        key = (parts[-3], parts[-2], parts[-1])
        if key in train_index:
            samples.append({"pid": int(row["person_id"]), "all_pairs": train_index[key]})

all_pids = sorted(list(set(s["pid"] for s in samples)))
random.shuffle(all_pids)
train_pids = set(all_pids[int(len(all_pids)*CFG.val_ratio):])
train_samples = [s for s in samples if s["pid"] in train_pids]
pid2label = {p: i for i, p in enumerate(sorted(train_pids))}

tf = T.Compose([
    T.Resize((CFG.img_size, CFG.img_size)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class TVRIDDataset(Dataset):
    def __init__(self, data, pid_map): self.data, self.pid_map = data, pid_map
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        s = self.data[idx]
        p = random.choice(s["all_pairs"])
        return tf(Image.open(p[0]).convert("RGB")), tf(Image.open(p[1]).convert("RGB")), self.pid_map.get(s["pid"], -1)

train_loader = DataLoader(TVRIDDataset(train_samples, pid2label), batch_size=CFG.batch_size, shuffle=True, num_workers=2)

model = TVRIDModelEffNetV2L(num_ids=len(pid2label)).to(CFG.device)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler("cuda")

print(f"🚀 Fixed! Training V2-L (Feat Dim: 640) on {CFG.device}...")
for epoch in range(1, CFG.epochs + 1):
    model.train()
    t_loss = 0
    optimizer.zero_grad()
    for i, (rgb, dep, target) in enumerate(train_loader):
        rgb, dep, target = rgb.to(CFG.device), dep.to(CFG.device), target.to(CFG.device)
        with autocast("cuda"):
            _, logits = model(rgb, dep)
            loss = criterion(logits, target) / CFG.accumulation_step
        scaler.scale(loss).backward()
        if (i + 1) % CFG.accumulation_step == 0:
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
        t_loss += loss.item() * CFG.accumulation_step
    print(f"Epoch {epoch:02d} | Loss: {t_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), "final_effnetv2l_model.pth")


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


🚀 Fixed! Training V2-L (Feat Dim: 640) on cuda...
Epoch 01 | Loss: 4.0354
Epoch 02 | Loss: 3.8448
Epoch 03 | Loss: 3.6537
Epoch 04 | Loss: 3.3616
Epoch 05 | Loss: 3.0784
Epoch 06 | Loss: 2.8318
Epoch 07 | Loss: 2.5395
Epoch 08 | Loss: 2.2681
Epoch 09 | Loss: 2.0614
Epoch 10 | Loss: 1.7840
Epoch 11 | Loss: 1.6671
Epoch 12 | Loss: 1.4648
Epoch 13 | Loss: 1.3672
Epoch 14 | Loss: 1.2439
Epoch 15 | Loss: 1.0835
Epoch 16 | Loss: 0.9856
Epoch 17 | Loss: 0.9212
Epoch 18 | Loss: 0.8328
Epoch 19 | Loss: 0.7827
Epoch 20 | Loss: 0.7014
Epoch 21 | Loss: 0.7326
Epoch 22 | Loss: 0.6995
Epoch 23 | Loss: 0.5840
Epoch 24 | Loss: 0.5527
Epoch 25 | Loss: 0.5322
Epoch 26 | Loss: 0.5762
Epoch 27 | Loss: 0.4228
Epoch 28 | Loss: 0.4907
Epoch 29 | Loss: 0.4257
Epoch 30 | Loss: 0.3762
Epoch 31 | Loss: 0.3855
Epoch 32 | Loss: 0.3753
Epoch 33 | Loss: 0.3141
Epoch 34 | Loss: 0.3142
Epoch 35 | Loss: 0.3223
Epoch 36 | Loss: 0.2800
Epoch 37 | Loss: 0.2950
Epoch 38 | Loss: 0.2612
Epoch 39 | Loss: 0.2315
Epoch 40 | Los

In [5]:
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
import pandas as pd
import zipfile
import os
from pathlib import Path

# ============================================================
# 1. BUILD TEST INDEX
# ============================================================
def build_test_index(base_path):
    index = {}
    base = Path(base_path)
    # Recursively find folders containing images in the test set
    for passage_dir in base.glob("**"):
        rgb_files = sorted(passage_dir.glob("*RGB*.png"))
        depth_map = {d.name.replace("_depth.png", ""): d
                     for d in passage_dir.glob("*depth*.png")}
        
        pairs = [(str(r), str(depth_map[r.name.replace("_RGB.png", "")]))
                 for r in rgb_files
                 if r.name.replace("_RGB.png", "") in depth_map]
        
        if pairs:
            # Use the folder name as gallery_id
            index[passage_dir.name] = pairs
    print(f"✅ Indexed {len(index)} test passages.")
    return index

# ============================================================
# 2. INFERENCE HELPERS (UPDATED FOR V2-L)
# ============================================================
@torch.no_grad()
def get_embedding(model, rgb_path, dep_path, device):
    model.eval()
    # Using the same normalization used during training
    tf = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    rgb = tf(Image.open(rgb_path).convert("RGB")).unsqueeze(0).to(device)
    dep = tf(Image.open(dep_path).convert("RGB")).unsqueeze(0).to(device)
    
    # Extract embedding from the TVRIDModelEffNetV2L structure
    emb = model.encode(rgb, dep)
    return F.normalize(emb, dim=1).cpu()

def generate_submission(model, test_index, device):
    all_embs = []
    g_ids = []
    g_paths = []
    
    print("🚀 Extracting embeddings for EfficientNetV2-L test set...")
    items = list(test_index.items())
    for i, (gid, pairs) in enumerate(items):
        passage_embs = []
        for rgb_p, dep_p in pairs:
            passage_embs.append(get_embedding(model, rgb_p, dep_p, device))
        
        # Average pooling across all frames in the passage to handle video sequences
        avg_emb = torch.cat(passage_embs).mean(0, keepdim=True)
        all_embs.append(avg_emb)
        g_ids.append(gid)
        g_paths.append(pairs[0][0]) 
        
        if (i+1) % 50 == 0:
            print(f"Processed {i+1}/{len(items)} passages")

    all_embs = torch.cat(all_embs)
    # Cosine Similarity Matrix: Efficient for Re-ID ranking
    sim_matrix = all_embs @ all_embs.T
    
    rows = []
    print("📊 Computing rankings...")
    for i, q_id in enumerate(g_ids):
        sims = sim_matrix[i].clone()
        sims[i] = -1e9 # Mask self-match
        
        indices = sims.argsort(descending=True)
        
        for rank, idx in enumerate(indices, 1):
            target_idx = idx.item()
            rows.append({
                "query_gallery_id": q_id,
                "query_path": g_paths[i],
                "gallery_id": g_ids[target_idx],
                "gallery_path": g_paths[target_idx],
                "rank": rank,
                "distance": round(float(1.0 - sims[target_idx]), 6)
            })

    # Save to CSV for Codabench submission
    out_csv = "rankings_rgb.csv"
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    
    with zipfile.ZipFile("submission_effnetv2l.zip", "w") as zf:
        zf.write(out_csv)
    print(f"\n✅ SUCCESS! Created submission_effnetv2l.zip")

# ============================================================
# 3. RUN
# ============================================================
# Ensure your model variable is the TVRIDModelEffNetV2L instance
test_path = "/kaggle/input/extracted-dataset/test_public"
test_index = build_test_index(test_path)

model.eval()
generate_submission(model, test_index, CFG.device)

✅ Indexed 204 test passages.
🚀 Extracting embeddings for EfficientNetV2-L test set...
Processed 50/204 passages
Processed 100/204 passages
Processed 150/204 passages
Processed 200/204 passages
📊 Computing rankings...

✅ SUCCESS! Created submission_effnetv2l.zip


In [6]:
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
import pandas as pd
import zipfile
from pathlib import Path

# ============================================================
# 1. ENHANCED INFERENCE HELPER
# ============================================================
@torch.no_grad()
def extract_embeddings(model, test_index, device):
    model.eval()
    tf = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    rgb_embeddings = []
    depth_embeddings = []
    ids = []
    paths = []

    print("🚀 Extracting Dual-Stream embeddings...")
    items = list(test_index.items())
    for i, (gid, pairs) in enumerate(items):
        p_rgb_embs = []
        p_dep_embs = []
        
        for rgb_p, dep_p in pairs:
            rgb_img = tf(Image.open(rgb_p).convert("RGB")).unsqueeze(0).to(device)
            dep_img = tf(Image.open(dep_p).convert("RGB")).unsqueeze(0).to(device)
            
            # Get raw features from the encoders before fusion
            # Using your specific model structure
            f_rgb = model.gem(model.rgb_enc(rgb_img)[-1]).flatten(1)
            f_dep = model.gem(model.dep_enc(dep_img)[-1]).flatten(1)
            
            p_rgb_embs.append(F.normalize(f_rgb, dim=1).cpu())
            p_dep_embs.append(F.normalize(f_dep, dim=1).cpu())
        
        # Mean pooling across passage frames
        rgb_embeddings.append(torch.cat(p_rgb_embs).mean(0, keepdim=True))
        depth_embeddings.append(torch.cat(p_dep_embs).mean(0, keepdim=True))
        ids.append(gid)
        paths.append(pairs[0][0])
        
        if (i+1) % 50 == 0: print(f"Processed {i+1}/{len(items)} passages")

    return torch.cat(rgb_embeddings), torch.cat(depth_embeddings), ids, paths

# ============================================================
# 2. TRIPLE RANKING GENERATOR
# ============================================================
def create_rankings(q_embs, g_embs, q_ids, g_ids, q_paths, g_paths, filename):
    sim_matrix = q_embs @ g_embs.T
    rows = []
    for i, q_id in enumerate(q_ids):
        sims = sim_matrix[i].clone()
        # Mask self-match only if searching within the same modality
        if q_ids == g_ids: sims[i] = -1e9 
        
        indices = sims.argsort(descending=True)
        for rank, idx in enumerate(indices, 1):
            target_idx = idx.item()
            rows.append({
                "query_gallery_id": q_id, "query_path": q_paths[i],
                "gallery_id": g_ids[target_idx], "gallery_path": g_paths[target_idx],
                "rank": rank, "distance": round(float(1.0 - sims[target_idx]), 6)
            })
    pd.DataFrame(rows).to_csv(filename, index=False)
    print(f"✅ Saved {filename}")

# ============================================================
# 3. EXECUTION
# ============================================================
rgb_embs, dep_embs, ids, paths = extract_embeddings(model, test_index, CFG.device)

# A. Generate RGB rankings (RGB Query vs RGB Gallery)
create_rankings(rgb_embs, rgb_embs, ids, ids, paths, paths, "rankings_rgb.csv")

# B. Generate Depth rankings (Depth Query vs Depth Gallery)
create_rankings(dep_embs, dep_embs, ids, ids, paths, paths, "rankings_depth.csv")

# C. Generate Cross-Modal rankings (RGB Query vs Depth Gallery)
create_rankings(rgb_embs, dep_embs, ids, ids, paths, paths, "rankings_cross.csv")

# Final Zip
with zipfile.ZipFile("submission_full_v2l.zip", "w") as zf:
    zf.write("rankings_rgb.csv")
    zf.write("rankings_depth.csv")
    zf.write("rankings_cross.csv")
print("\n📦 Final Zip created! Upload 'submission_full_v2l.zip' to fill all columns.")

🚀 Extracting Dual-Stream embeddings...
Processed 50/204 passages
Processed 100/204 passages
Processed 150/204 passages
Processed 200/204 passages
✅ Saved rankings_rgb.csv
✅ Saved rankings_depth.csv
✅ Saved rankings_cross.csv

📦 Final Zip created! Upload 'submission_full_v2l.zip' to fill all columns.
